# Music Annotation Project V7.2.1: Optimised Phase B and Complete Reporting

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook starts after V7.1 Phase A. It does not run reconstruction and does not alter V7.1 reconstruction fingerprints or NPZ contents.

It:
- discovers and validates the existing V7.1 Phase-A state;
- stages clean and observed pages locally;
- reuses all valid V7.1 reconstruction NPZ files;
- evaluates each method with precomputed SSIM, ink, edge, difference and tolerance maps;
- checkpoints every page/dilation evaluation atomically;
- produces summaries, paired bootstrap comparisons, contact sheets, failure crops, a production decision table, a manifest and a resume audit.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration, validation and local staging


In [ ]:
import cv2, json, hashlib, shutil, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.metrics import structural_similarity as ssim

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'
OBSERVED_DIR=ROOT/'aligned'
OUTPUT_ROOT=ROOT/'reconstruction_benchmark_v7_1_real'
CACHE=OUTPUT_ROOT/'persistent_cache'
RECON_CACHE=CACHE/'reconstruction'
METRIC_CACHE=CACHE/'evaluation'
STATE_PATH=OUTPUT_ROOT/'benchmark_state.json'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/'runs'/(RUN_NAME+'_v7_2_1_complete')
TABLES=OUT/'tables'; CONTACTS=OUT/'contact_sheets'; FAILURES=OUT/'failure_crops'
LOCAL=Path('/content/reconstruction_benchmark_v7_2_1')
LCLEAN=LOCAL/'clean'; LOBSERVED=LOCAL/'observed'
for d in [OUTPUT_ROOT,CACHE,RECON_CACHE,METRIC_CACHE,OUT,TABLES,CONTACTS,FAILURES,LCLEAN,LOBSERVED]:
    d.mkdir(parents=True,exist_ok=True)

DILATIONS=[1,2]
MIN_METRIC_COMPONENT_AREA=20
BOOTSTRAP_SAMPLES=2000
MAX_CONTACT_PAGES=24
MAX_FAILURE_CROPS=20
METHODS=['biharmonic','telea','paper_fill','paper_biharmonic_hybrid','paper_telea_hybrid']
BASELINE='biharmonic'
FORCE_REBUILD_EVALUATION=False
EVALUATION_SCHEMA='v7.2.1_precomputed_feature_maps'

def image_files(folder):
    folder=Path(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []
def maps(folder): return {p.stem:p for p in image_files(folder)}
def find_local(folder,page):
    hits=[p for p in image_files(folder) if p.stem.lower()==page.lower()]
    return hits[0] if len(hits)==1 else None
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def object_hash(value): return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()
def read_json(path,default):
    try: return json.loads(Path(path).read_text()) if Path(path).exists() else default
    except Exception: return default
def write_json_atomic(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,allow_nan=True,default=str)); tmp.replace(path)
def valid_npz(path,required):
    try:
        with np.load(path) as z: return set(required).issubset(z.files)
    except Exception: return False
def copy_current(source,destination):
    destination=Path(destination)
    if not destination.exists() or md5(source)!=md5(destination): shutil.copy2(source,destination)

state=read_json(STATE_PATH,None)
if not state or not isinstance(state.get('items'),dict):
    raise RuntimeError(f'V7.1 state is missing or invalid: {STATE_PATH}')
state.setdefault('failures',[])

clean_map=maps(CLEAN_DIR); observed_map=maps(OBSERVED_DIR)
reconstructed_items={k:v for k,v in state['items'].items() if v.get('reconstruction_complete')}
valid_pages=[]; issues=[]
for page in sorted({v.get('page') for v in reconstructed_items.values() if v.get('page')}):
    cp=clean_map.get(page); op=observed_map.get(page)
    if cp is None or op is None:
        issues.append({'page':page,'reason':'clean or observed source missing'}); continue
    clean=cv2.imread(str(cp),0); observed=cv2.imread(str(op),0)
    if clean is None or observed is None:
        issues.append({'page':page,'reason':'clean or observed source unreadable'}); continue
    if clean.shape!=observed.shape:
        issues.append({'page':page,'reason':'clean/observed shape mismatch'}); continue
    complete=True
    for dilation in DILATIONS:
        item=state['items'].get(f'{page}|d{dilation}',{})
        cache_path=Path(item.get('reconstruction_cache',''))
        if not item.get('reconstruction_complete') or not valid_npz(cache_path,['mask',*METHODS]):
            issues.append({'page':page,'dilation':dilation,'reason':'valid Phase-A reconstruction cache missing'})
            complete=False
    if complete:
        copy_current(cp,LCLEAN/(page+cp.suffix.lower()))
        copy_current(op,LOBSERVED/(page+op.suffix.lower()))
        valid_pages.append(page)

pd.DataFrame(issues).to_csv(TABLES/'discovery_issues.csv',index=False)
if not valid_pages: raise RuntimeError('No complete V7.1 Phase-A page/dilation cache pairs were found.')
discovered=valid_pages.copy()
print('Reusable Phase-A pages:',len(valid_pages),'Issues:',len(issues),'State:',STATE_PATH)


## Optimised evaluation functions


In [ ]:
def printed_ink(image):
    return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge_map(image): return cv2.Canny(image,70,170)>0
def classify_gt(clean_ink,mask):
    long_lines=cv2.morphologyEx(clean_ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long_lines.astype(np.uint8),np.ones((3,7),np.uint8))>0
    near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0
    m=mask>0
    return {'all':m,'background':m&~clean_ink,'staff':m&staff,'symbol':m&clean_ink&~near,'mixed':m&clean_ink&near}
def tolerance_maps(reference,prediction,tolerance=1):
    kernel=np.ones((2*tolerance+1,2*tolerance+1),np.uint8)
    reference_dilated=cv2.dilate(reference.astype(np.uint8),kernel)>0
    prediction_dilated=cv2.dilate(prediction.astype(np.uint8),kernel)>0
    return {'reference':reference,'prediction':prediction,'prediction_match':prediction&reference_dilated,'reference_match':reference&prediction_dilated}
def feature_bundle(clean,reconstruction,clean_ink,clean_edges):
    signed=clean.astype(np.float32)-reconstruction.astype(np.float32)
    _,ssim_map=ssim(clean,reconstruction,data_range=255,full=True)
    return {'abs_diff':np.abs(signed),'squared_diff':signed*signed,'ssim_map':ssim_map.astype(np.float32,copy=False),
            'ink':tolerance_maps(clean_ink,printed_ink(reconstruction)),'edge':tolerance_maps(clean_edges,edge_map(reconstruction)),
            'dark_smudge':(reconstruction<180)&(clean>=220),'clean_light':clean>=220}
def f1_from_maps(feature_maps,region):
    reference_count=int(np.sum(feature_maps['reference']&region))
    if reference_count==0: return np.nan,np.nan,np.nan
    prediction_count=int(np.sum(feature_maps['prediction']&region))
    precision=float(np.sum(feature_maps['prediction_match']&region)/max(prediction_count,1))
    recall=float(np.sum(feature_maps['reference_match']&region)/reference_count)
    return precision,recall,float(2*precision*recall/max(precision+recall,1e-12))
def score(bundle,region):
    pixels=int(np.sum(region))
    if pixels==0: return None
    mse=float(np.mean(bundle['squared_diff'][region]))
    ink_precision,ink_recall,ink_f1=f1_from_maps(bundle['ink'],region)
    _,_,edge_f1=f1_from_maps(bundle['edge'],region)
    return {'pixels':pixels,'mae':float(np.mean(bundle['abs_diff'][region])),
            'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),
            'ssim':float(np.mean(bundle['ssim_map'][region])),
            'ink_precision':ink_precision,'ink_recall':ink_recall,'ink_f1':ink_f1,
            'edge_f1_tolerant':edge_f1,
            'dark_smudge_rate':float(np.sum(bundle['dark_smudge']&region)/max(np.sum(bundle['clean_light']&region),1))}
def crop_bundle(bundle,x,y,w,h):
    return {'abs_diff':bundle['abs_diff'][y:y+h,x:x+w],
            'squared_diff':bundle['squared_diff'][y:y+h,x:x+w],
            'ssim_map':bundle['ssim_map'][y:y+h,x:x+w],
            'dark_smudge':bundle['dark_smudge'][y:y+h,x:x+w],
            'clean_light':bundle['clean_light'][y:y+h,x:x+w],
            'ink':{k:v[y:y+h,x:x+w] for k,v in bundle['ink'].items()},
            'edge':{k:v[y:y+h,x:x+w] for k,v in bundle['edge'].items()}}


## Phase B: atomic optimised evaluation checkpoints


In [ ]:
metric_rows=[]; component_rows=[]
for page_index,page in enumerate(valid_pages,1):
    cp=find_local(LCLEAN,page); op=find_local(LOBSERVED,page)
    clean=cv2.imread(str(cp),0); observed=cv2.imread(str(op),0)
    clean_ink=printed_ink(clean); clean_edges=edge_map(clean)
    for dilation in DILATIONS:
        item_id=f'{page}|d{dilation}'; item=state['items'][item_id]
        evaluation_fingerprint=object_hash({'reconstruction':item['fingerprint'],'clean_md5':md5(cp),'evaluation_schema':EVALUATION_SCHEMA,
            'minimum_component':MIN_METRIC_COMPONENT_AREA,'ink':[41,15],'edge':[70,170],'tolerance':1,'smudge':[180,220]})
        evaluation_cache=METRIC_CACHE/f'{page}_d{dilation}_{evaluation_fingerprint}.json'
        payload=read_json(evaluation_cache,None) if evaluation_cache.exists() and not FORCE_REBUILD_EVALUATION else None
        hit=bool(payload and payload.get('evaluation_fingerprint')==evaluation_fingerprint and payload.get('evaluation_schema')==EVALUATION_SCHEMA)
        try:
            if not hit:
                reconstruction_cache=Path(item['reconstruction_cache'])
                with np.load(reconstruction_cache) as z:
                    mask=z['mask']; outputs={method:z[method] for method in METHODS}
                regions=classify_gt(clean_ink,mask)
                count,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
                component_specs=[]
                for component in range(1,count):
                    area=int(stats[component,cv2.CC_STAT_AREA])
                    if area<MIN_METRIC_COMPONENT_AREA: continue
                    x,y,w,h=map(int,stats[component,:4])
                    local_mask=labels[y:y+h,x:x+w]==component
                    ink_fraction=float(np.mean(clean_ink[y:y+h,x:x+w][local_mask]))
                    component_specs.append({'component':component,'category':'background' if ink_fraction<.05 else ('structure' if ink_fraction>.30 else 'mixed'),
                        'component_area':area,'x':x,'y':y,'w':w,'h':h,'aspect_ratio':max(w,h)/max(min(w,h),1),
                        'clean_ink_fraction':ink_fraction,'local_mask':local_mask})
                local_metrics=[]; local_components=[]
                for method,reconstruction in outputs.items():
                    if not np.array_equal(reconstruction[mask==0],observed[mask==0]):
                        raise RuntimeError(f'{page} d{dilation} {method} changed unmasked pixels')
                    bundle=feature_bundle(clean,reconstruction,clean_ink,clean_edges)
                    for region_name,region in regions.items():
                        values=score(bundle,region)
                        if values: local_metrics.append({'page':page,'dilation':dilation,'method':method,'region':region_name,**values})
                    for spec in component_specs:
                        local_bundle=crop_bundle(bundle,spec['x'],spec['y'],spec['w'],spec['h'])
                        values=score(local_bundle,spec['local_mask'])
                        if values:
                            public_spec={k:v for k,v in spec.items() if k!='local_mask'}
                            local_components.append({'page':page,'dilation':dilation,'method':method,**public_spec,**values})
                payload={'version':'7.2.1','evaluation_schema':EVALUATION_SCHEMA,'evaluation_fingerprint':evaluation_fingerprint,
                    'reconstruction_fingerprint':item['fingerprint'],'page':page,'dilation':dilation,
                    'metric_rows':local_metrics,'component_rows':local_components,'created_utc':datetime.now(timezone.utc).isoformat()}
                write_json_atomic(evaluation_cache,payload)
            metric_rows.extend(payload['metric_rows']); component_rows.extend(payload['component_rows'])
            state['items'][item_id].update({'evaluation_complete':True,'evaluation_schema':EVALUATION_SCHEMA,
                'evaluation_fingerprint':evaluation_fingerprint,'evaluation_cache':str(evaluation_cache),
                'updated_utc':datetime.now(timezone.utc).isoformat()})
            write_json_atomic(STATE_PATH,state)
            print(f'Phase B {page_index}/{len(valid_pages)} {page} d{dilation}:','HIT' if hit else 'MISS')
        except Exception as error:
            state['items'][item_id].update({'evaluation_complete':False,'evaluation_schema':EVALUATION_SCHEMA,'evaluation_error':str(error)})
            state['failures'].append({'phase':'B-v7.2.1','page':page,'dilation':dilation,'error':str(error),'utc':datetime.now(timezone.utc).isoformat()})
            write_json_atomic(STATE_PATH,state); print('FAILED',item_id,error)

results=pd.DataFrame(metric_rows); components=pd.DataFrame(component_rows)
if results.empty: raise RuntimeError('No evaluation records available.')
results.to_csv(TABLES/'metrics_by_page_region.csv',index=False)
components.to_csv(TABLES/'component_metrics.csv',index=False)
print('Phase B complete. Metric rows:',len(results),'Component rows:',len(components))


## Summaries and paired bootstrap comparisons


In [ ]:
summary=(results.groupby(['dilation','method','region'],as_index=False).agg(
    pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),psnr=('psnr','mean'),ssim=('ssim','mean'),
    ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1_tolerant=('edge_f1_tolerant','mean'),dark_smudge_rate=('dark_smudge_rate','mean')))
summary.to_csv(TABLES/'summary_by_region.csv',index=False)
def bootstrap(values,seed=42):
    values=np.asarray(values,float); values=values[np.isfinite(values)]
    if not len(values): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed)
    samples=np.array([rng.choice(values,len(values),replace=True).mean() for _ in range(BOOTSTRAP_SAMPLES)])
    return float(values.mean()),float(np.percentile(samples,2.5)),float(np.percentile(samples,97.5))
paired=[]
for dilation in DILATIONS:
    for region,metric,higher in [('all','mae',False),('all','ssim',True),('background','mae',False),('background','dark_smudge_rate',False),('staff','ink_recall',True),('symbol','edge_f1_tolerant',True),('mixed','edge_f1_tolerant',True)]:
        pivot=results[(results.dilation==dilation)&(results.region==region)].pivot(index='page',columns='method',values=metric)
        if BASELINE not in pivot: continue
        for method in METHODS:
            if method==BASELINE or method not in pivot: continue
            delta=(pivot[method]-pivot[BASELINE])*(1 if higher else -1)
            mean,low,high=bootstrap(delta.dropna())
            paired.append({'dilation':dilation,'region':region,'metric':metric,'method':method,'baseline':BASELINE,
                'benefit_mean':mean,'ci95_low':low,'ci95_high':high,'wins':int((delta>0).sum()),'losses':int((delta<0).sum()),'ties':int((delta==0).sum())})
paired_df=pd.DataFrame(paired)
paired_df.to_csv(TABLES/'paired_method_deltas.csv',index=False)
display(summary); display(paired_df)


## Contact sheets and worst background-smudge crops


In [ ]:
def load_item(page,dilation):
    clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBSERVED,page)),0)
    item=state['items'][f'{page}|d{dilation}']
    with np.load(item['reconstruction_cache']) as z: mask=z['mask']; outputs={method:z[method] for method in METHODS}
    return clean,observed,mask,outputs

sizes=[]
for page in valid_pages:
    item=state['items'][f'{page}|d{max(DILATIONS)}']
    with np.load(item['reconstruction_cache']) as z: sizes.append({'page':page,'mask_pixels':int(np.sum(z['mask']>0))})
sizes=pd.DataFrame(sizes).sort_values('mask_pixels',ascending=False)
for page in sizes.head(MAX_CONTACT_PAGES).page:
    dilation=max(DILATIONS); clean,observed,mask,outputs=load_item(page,dilation)
    overlay=cv2.cvtColor(observed,cv2.COLOR_GRAY2BGR); layer=overlay.copy(); layer[mask>0]=(0,0,255); overlay=cv2.addWeighted(overlay,.68,layer,.32,0)
    images=[clean,observed,overlay]+[outputs[m] for m in METHODS]; names=['clean','observed','mask']+METHODS
    fig,axes=plt.subplots(2,4,figsize=(24,14))
    for ax,image,name in zip(axes.ravel(),images,names):
        ax.imshow(image if image.ndim==2 else cv2.cvtColor(image,cv2.COLOR_BGR2RGB),cmap='gray' if image.ndim==2 else None,vmin=0 if image.ndim==2 else None,vmax=255 if image.ndim==2 else None)
        ax.set_title(name); ax.axis('off')
    for ax in axes.ravel()[len(images):]: ax.axis('off')
    plt.tight_layout(); plt.savefig(CONTACTS/f'{page}_d{dilation}.png',dpi=150,bbox_inches='tight'); plt.close(fig)

if not components.empty:
    worst=components[(components.method==BASELINE)&(components.category=='background')].sort_values(['dark_smudge_rate','component_area'],ascending=[False,False]).head(MAX_FAILURE_CROPS)
    for rank,(_,row) in enumerate(worst.iterrows(),1):
        page=str(row.page); dilation=int(row.dilation); clean,observed,mask,outputs=load_item(page,dilation); height,width=clean.shape; pad=32
        x0=max(0,int(row.x)-pad); y0=max(0,int(row.y)-pad); x1=min(width,int(row.x+row.w)+pad); y1=min(height,int(row.y+row.h)+pad)
        panels=[clean[y0:y1,x0:x1],observed[y0:y1,x0:x1]]+[outputs[m][y0:y1,x0:x1] for m in METHODS]
        names=['clean','observed']+METHODS; fig,axes=plt.subplots(1,len(panels),figsize=(4*len(panels),4))
        for ax,image,name in zip(axes,panels,names): ax.imshow(image,cmap='gray',vmin=0,vmax=255); ax.set_title(name); ax.axis('off')
        plt.tight_layout(); plt.savefig(FAILURES/f'{rank:03d}_{page}_d{dilation}_c{int(row.component)}.png',dpi=170,bbox_inches='tight'); plt.close(fig)
print('Contact sheets:',CONTACTS,'Failure crops:',FAILURES)


## Decision table, manifest and resume audit


In [ ]:
decision=[]
for dilation in DILATIONS:
    for method in METHODS:
        if method==BASELINE: continue
        def select(region,metric):
            return paired_df[(paired_df.dilation==dilation)&(paired_df.region==region)&(paired_df.metric==metric)&(paired_df.method==method)]
        mae=select('all','mae'); smudge=select('background','dark_smudge_rate'); mixed=select('mixed','edge_f1_tolerant')
        decision.append({'dilation':dilation,'method':method,
            'all_mae_benefit':float(mae.iloc[0].benefit_mean) if len(mae) else np.nan,
            'all_mae_ci_low':float(mae.iloc[0].ci95_low) if len(mae) else np.nan,
            'background_smudge_benefit':float(smudge.iloc[0].benefit_mean) if len(smudge) else np.nan,
            'background_smudge_ci_low':float(smudge.iloc[0].ci95_low) if len(smudge) else np.nan,
            'mixed_edge_benefit':float(mixed.iloc[0].benefit_mean) if len(mixed) else np.nan,
            'candidate_for_production':bool(len(mae) and len(smudge) and mae.iloc[0].benefit_mean>0 and smudge.iloc[0].benefit_mean>0)})
decision_df=pd.DataFrame(decision).sort_values(['candidate_for_production','background_smudge_benefit','all_mae_benefit'],ascending=[False,False,False])
decision_df.to_csv(TABLES/'production_decision_table.csv',index=False)
status=pd.DataFrame([{'item_id':key,**value} for key,value in state['items'].items()])
status.to_csv(TABLES/'resume_status.csv',index=False)
settings={'source_phase':'V7.1 Phase A','evaluation_schema':EVALUATION_SCHEMA,'methods':METHODS,'dilations':DILATIONS,
          'minimum_metric_component_area':MIN_METRIC_COMPONENT_AREA,'bootstrap_samples':BOOTSTRAP_SAMPLES}
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.2.1','pages_discovered':len(discovered),
          'pages_evaluated':int(results.page.nunique()),'methods':METHODS,'dilations':DILATIONS,'settings':settings,
          'guarantees':{'v7_1_phase_a_cache_compatible':True,'clean_not_passed_to_reconstruction':True,
              'outside_mask_immutable_checked':True,'atomic_evaluation_checkpoint':True,'precomputed_phase_b_feature_maps':True},
          'resume':{'state':str(STATE_PATH),'reconstruction_cache':str(RECON_CACHE),'evaluation_cache':str(METRIC_CACHE),
              'tracked_items':len(state['items']),'reconstruction_complete':int(status.reconstruction_complete.fillna(False).sum()),
              'evaluation_complete':int(status.evaluation_complete.fillna(False).sum())}}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2))
state['last_completed_run']=str(OUT); state['updated_utc']=datetime.now(timezone.utc).isoformat(); write_json_atomic(STATE_PATH,state)
display(decision_df)
print('V7.2.1 COMPLETE:',OUT)
print('Persistent state:',STATE_PATH)
